In [ ]:
import sys, os, json, hashlib, shutil, zipfile
from pathlib import Path
COMP = Path('/kaggle/input/competitions/google-verifiable-quantum-circuit-compilation')
WORK = Path('/kaggle/working')
sys.path.insert(0, str(COMP / 'kickmix'))
sys.path.insert(0, str(COMP / 'verifiers'))
from invsqrt import verify_submission_invsqrt as IV
from dqi import verify_submission_dqi as DV
from dfthcsos.verify import verify as CV

# Extract submission.zip
sub_dir = WORK / 'submission'
shutil.rmtree(sub_dir, ignore_errors=True)
sub_dir.mkdir()
from pathlib import Path
zip_path = None
for p in Path('/kaggle/input').rglob('*.zip'):
    zip_path = p
    break
print('Found zip at:', zip_path)
import os
print('Input dir contents:', os.listdir('/kaggle/input'))
with zipfile.ZipFile(zip_path) as z:
    z.extractall(sub_dir)

print('Verifying invsqrt...')
c = __import__('kickmix').Circuit.from_file(str(sub_dir / 'invsqrt.kmb'), format='kmb')
r_inv = IV.verify(c, IV.ProblemSpecInvSqrt.for_n_p(10), max_workers=4)
print('invsqrt:', r_inv['status'], r_inv['score'])

print('Verifying dqi...')
c = __import__('kickmix').Circuit.from_file(str(sub_dir / 'dqi.kmb'), format='kmb')
r_dqi = DV.verify(c, DV.ProblemSpecDQI.benchmark_instance(num_shots=102400), max_workers=4)
print('dqi:', r_dqi['status'], r_dqi['score'])

print('Verifying dfthcsos...')
r_chem = CV(sub_dir / 'dfthcsos')
print('dfthcsos:', r_chem['status'], r_chem['score'])

gm = (r_inv['score'] / 2076774.233715282 * r_dqi['score'] / 24439473914136.668 * r_chem['score'] / 3355599588.3155365) ** (1/3)
print('Geometric Mean:', gm)

report = {'invsqrt': r_inv, 'dqi': r_dqi, 'dfthcsos': r_chem, 'geometric_mean': gm}
(WORK / 'validation_report.json').write_text(json.dumps(report, indent=2, default=float))
